# Pinch Analysis: Crude Preheat Train
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. `PinchWorkspace` keeps named study cases, while each case remains a real `PinchProblem` with the familiar `target`, `plot`, `summary_frame`, and `compare_to` workflow.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from OpenPinch import PinchWorkspace

In [ ]:
workspace = PinchWorkspace(
    source="crude_preheat_train.json",
    project_name="crude_preheat_train",
)
baseline = workspace.use_case("baseline")

summary = baseline.summary_frame()
catalog = baseline.plot.catalog()

{
    "cases": workspace.list_cases(),
    "active_case": workspace.active_case_name,
    "graph_count": len(catalog),
    "zone_tree_present": "zone_tree" in workspace.get_case_payload("baseline"),
}

In [ ]:
summary

## Baseline graphs from the built-in plot accessor
Because each workspace case is a real `PinchProblem`, the notebook can use the normal `plot` accessor directly instead of rebuilding figures from serialized helper payloads.


In [ ]:
catalog.loc[
    catalog["Target"] == "crude_preheat_train/Direct Integration",
    ["Target", "Graph Type", "Graph Name"],
].reset_index(drop=True)

In [ ]:
baseline.plot.composite_curve(zone_name="crude_preheat_train")

In [ ]:
baseline.plot.shifted_composite_curve(
    zone_name="crude_preheat_train",
)

In [ ]:
baseline.plot.grand_composite_curve(
    zone_name="crude_preheat_train/Direct Integration",
)

## One named case before the sweep
Clone the baseline into a named case, adjust the root `dt_cont_multiplier`, and compare the two `PinchProblem` summaries directly.


In [ ]:
wide_dt = workspace.copy_case("baseline", "wide_dt", activate=False)
wide_dt.set_dt_cont_multiplier(2.0)
workspace.compare_cases(
    "baseline",
    "wide_dt",
    target_name="crude_preheat_train/Direct Integration",
)

In [ ]:
rows = []
for factor in np.linspace(0.5, 10.0, 96):
    case_name = f"dt_mult_{factor:05.2f}".replace(".", "_")
    case = workspace.copy_case("baseline", case_name, activate=False)
    case.set_dt_cont_multiplier(float(factor))
    row = (
        case.summary_frame()
        .loc[lambda frame: frame["Target"] == "crude_preheat_train/Direct Integration"]
        .iloc[0]
    )
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity

In [ ]:
series_colors = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
    "Hot Pinch": "#d68910",
    "Cold Pinch": "#7d3c98",
}

sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
sensitivity_fig